In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

In [2]:
train = pd.read_csv(r"C:\Users\hecto\Documents\PythonProjects\obesity_risk\train.csv")

train.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
#prep X and y
target = "NObeyesdad"

X = train.drop(columns=["id", target])
y = train[target]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [8]:
#id variable types
categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
Numeric: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


In [9]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

In [10]:
#preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

Baseline Logistic Regression

In [11]:
log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        multi_class="multinomial"
    ))
])

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

log_acc = accuracy_score(y_test, log_pred)

print("Logistic Regression Accuracy:", log_acc)

C:\ProgramData\anaconda3\envs\ds_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression Accuracy: 0.8692196531791907


In [12]:
print(classification_report(
    y_test,
    log_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.89      0.95      0.92       505
      Normal_Weight       0.87      0.82      0.85       617
     Obesity_Type_I       0.81      0.85      0.83       582
    Obesity_Type_II       0.93      0.96      0.95       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.75      0.71      0.73       485
Overweight_Level_II       0.73      0.71      0.72       504

           accuracy                           0.87      4152
          macro avg       0.85      0.86      0.85      4152
       weighted avg       0.87      0.87      0.87      4152



## Multinomial Logistic Regression Interpretation

The multinomial logistic regression model achieved approximately 86.9% classification accuracy on the obesity risk dataset. The model performed well overall, particularly for the severe obesity categories, but showed weaker performance for the overweight categories where class overlap was more substantial. Logistic regression assumes a linear relationship between the predictors and the log-odds of each obesity category. The lower predictive performance compared to the ensemble methods suggests that the dataset contains nonlinear relationships and interactions that cannot be fully captured by a linear decision boundary. Despite these limitations, the model provided a strong baseline for comparing more advanced classification approaches.

Bagging

In [13]:
bag_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42
    ))
])

bag_model.fit(X_train, y_train)

bag_pred = bag_model.predict(X_test)

bag_acc = accuracy_score(y_test, bag_pred)

print("Bagging Accuracy:", bag_acc)

Bagging Accuracy: 0.8894508670520231


In [14]:
print(classification_report(
    y_test,
    bag_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.93      0.94      0.93       505
      Normal_Weight       0.86      0.86      0.86       617
     Obesity_Type_I       0.87      0.88      0.87       582
    Obesity_Type_II       0.96      0.97      0.97       650
   Obesity_Type_III       1.00      0.99      0.99       809
 Overweight_Level_I       0.76      0.72      0.74       485
Overweight_Level_II       0.77      0.78      0.78       504

           accuracy                           0.89      4152
          macro avg       0.88      0.88      0.88      4152
       weighted avg       0.89      0.89      0.89      4152



## Bagging Model Interpretation

The bagging classifier improved overall predictive performance to approximately 88.9% accuracy. Bagging works by fitting multiple decision trees on bootstrap samples of the training data and averaging their predictions to reduce variance and improve stability. Compared to logistic regression, the bagging model produced stronger classification performance across several obesity categories, particularly for the overweight and moderate obesity groups. The improvement indicates that nonlinear relationships and variable interactions are present within the dataset and are better captured through ensemble tree-based methods than through a single linear classification boundary.

Random Forest

In [15]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.8966763005780347


In [16]:
print(classification_report(
    y_test,
    rf_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.93      0.93      0.93       505
      Normal_Weight       0.84      0.88      0.86       617
     Obesity_Type_I       0.88      0.88      0.88       582
    Obesity_Type_II       0.97      0.97      0.97       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.81      0.72      0.76       485
Overweight_Level_II       0.79      0.81      0.80       504

           accuracy                           0.90      4152
          macro avg       0.89      0.89      0.89      4152
       weighted avg       0.90      0.90      0.90      4152



## Random Forest Interpretation

The random forest classifier achieved approximately 89.7% accuracy, outperforming both multinomial logistic regression and bagging. Random forests extend bagging by introducing random subsets of predictors at each split, which reduces correlation among trees and improves generalization performance. The model demonstrated strong predictive accuracy across nearly all obesity categories and substantially improved performance for the overweight classifications. The results suggest that random forests effectively captured nonlinear patterns, interaction effects, and complex decision boundaries within the obesity dataset while also reducing the risk of overfitting.

Using XGBoost instead of BART

In [17]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=len(label_encoder.classes_),
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)

print("XGBoost Accuracy:", xgb_acc)

XGBoost Accuracy: 0.9096820809248555


In [18]:
print(classification_report(
    y_test,
    xgb_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.94      0.95      0.94       505
      Normal_Weight       0.88      0.90      0.89       617
     Obesity_Type_I       0.89      0.90      0.89       582
    Obesity_Type_II       0.97      0.97      0.97       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.81      0.78      0.79       485
Overweight_Level_II       0.81      0.81      0.81       504

           accuracy                           0.91      4152
          macro avg       0.90      0.90      0.90      4152
       weighted avg       0.91      0.91      0.91      4152



## XGBoost Interpretation

The XGBoost classifier produced the strongest overall performance with approximately 91.0% classification accuracy. XGBoost is a boosting algorithm that sequentially builds decision trees, where each new tree attempts to correct errors made by previous trees. The model achieved excellent precision, recall, and F1-scores across nearly all obesity categories, including perfect classification performance for the Obesity_Type_III category. The superior performance of XGBoost demonstrates its ability to model complex nonlinear relationships, interaction effects, and difficult classification boundaries. These results indicate that boosted ensemble methods are highly effective for multi-class obesity prediction problems and outperform simpler linear classification approaches.

XGBoost was used instead of BART because it provides similar boosting-based ensemble capabilities while offering easier implementation, faster computation, and better integration within the Python machine learning ecosystem.